# Notebook 6 — Diagnostic Analysis: Why Did Sequential Editing Fail?
**CS 590NN | Amogh | Apr 24 — post-hoc diagnostic on v3 null result**

## Goal

Notebook 5 v3 reported null results on all three pre-registered hypotheses (H1, H2, H3). This notebook digs into the existing data to explain *why*, and to extract any defensible findings hiding underneath the null.

## What this notebook does

Five diagnostic analyses, each pointing at a different possible explanation for the null:

1. **Solo-edit baseline check.** Did edit A even succeed in the first place? If `p_new_A_postA` was already low, retention is meaningless.
2. **Circuit-size predictors of retention.** Is retention predicted by `n_mlp_A` or `n_mlp_B`? Larger circuits = more edit damage.
3. **Cross-method consistency.** When ACDC fails on a pair, do ROMEtrace and Random fail on the same pair? If yes, it's the *pair* that's hard, not the *method*.
4. **Negative KL super-additivity decomposition.** Why are sequential edits canceling rather than compounding?
5. **Bimodal retention pattern.** The retention distribution is heavily bimodal (mostly 0, sometimes 1). What separates the two modes?

## Inputs

Just one file: `sequential_edit_rows_v3_<ts>.json` from your Notebook 5 v3 results folder. No GPU needed.

### 6.0 Setup

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from pathlib import Path
from datetime import datetime

RESULTS_DIR = Path("results_nb6")
RESULTS_DIR.mkdir(exist_ok=True)
FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(exist_ok=True)

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", None)
print("Setup done.")

### 6.1 Load v3 Results

Upload `sequential_edit_rows_v3_*.json` from Notebook 5 v3.

In [ ]:
from google.colab import files
print("Upload sequential_edit_rows_v3_*.json")
uploaded = files.upload()
fname = list(uploaded.keys())[0]
with open(fname) as f:
    data = json.load(f)
rows = data['rows']
df = pd.DataFrame([r for r in rows if r.get('status') == 'ok'])
print(f"\nLoaded {len(df)} OK rows from {fname}")
print(f"Conditions: {df['condition'].unique().tolist()}")
print(f"Methods:    {df['method'].unique().tolist()}")
print(f"Pairs:      {df.groupby(['condition'])['pair_idx'].nunique().to_dict()}")

### 6.2 Diagnostic 1 — Did edit A even succeed in the first place?

**Logic:** Retention is `p_new_A_postAB / p_new_A_postA`. If the denominator was tiny, the ratio is meaningless. Likewise, `retention_A_abs = p_new_A_postAB` only carries signal if A actually flipped on the first edit.

**Test:** Drop pairs where `p_new_A_postA < 0.5` (edit A didn't even succeed) and rerun the H1 test on the remaining clean subset.

In [ ]:
# Distribution of edit-A success
print("Edit-A success distribution (before sequential edit):")
print(df.groupby(['condition','method'])['p_new_A_postA'].agg(['mean','median','min','max']).round(3))

# Plot
fig, ax = plt.subplots(figsize=(8, 4))
for i, (cond, sub) in enumerate(df.groupby('condition')):
    ax.hist(sub['p_new_A_postA'], bins=20, alpha=0.6, label=cond)
ax.set_xlabel('p_new_A_postA  (probability of target_new on A after edit A only)')
ax.set_ylabel('count'); ax.set_title('Did edit A succeed? Distribution across all rows')
ax.axvline(0.5, color='red', ls='--', label='success threshold = 0.5')
ax.legend(); fig.tight_layout(); fig.savefig(FIG_DIR / 'd1_edit_A_success.png', dpi=140); plt.show()

n_total = len(df)
n_success = (df['p_new_A_postA'] > 0.5).sum()
print(f"\n{n_success}/{n_total} ({100*n_success/n_total:.1f}%) rows have edit-A success > 0.5")

# Re-run H1 on clean subset
clean = df[df['p_new_A_postA'] > 0.5].copy()
if len(clean) > 10:
    s_ret = clean[clean['condition']=='shared']['retention_A_abs'].values
    d_ret = clean[clean['condition']=='disjoint']['retention_A_abs'].values
    if len(s_ret) > 5 and len(d_ret) > 5:
        u, p = stats.mannwhitneyu(s_ret, d_ret, alternative='less')
        print(f"\nH1 RE-TEST on clean subset (only rows where edit A actually succeeded):")
        print(f"  shared mean retention = {s_ret.mean():.3f} (n={len(s_ret)})")
        print(f"  disjoint mean retention = {d_ret.mean():.3f} (n={len(d_ret)})")
        print(f"  Mann-Whitney p (one-sided shared<disjoint) = {p:.4f}")
        if p < 0.05:
            print("  *** H1 RECOVERED on clean subset. Tells us: original null was a measurement-floor artifact. ***")

### 6.3 Diagnostic 2 — Is retention predicted by circuit size?

**Hypothesis:** Larger circuits at edit A and/or edit B → more parameter overlap → more wipe-out. If true, `retention_A_abs` should anti-correlate with `n_mlp_A + n_mlp_B`.

If circuit size predicts retention better than condition or method, that's the *real* explanatory variable — and it's something none of our pre-registered hypotheses tested for.

In [ ]:
df['n_mlp_total'] = df['n_mlp_A'] + df['n_mlp_B']
df['n_attn_total'] = df['n_attn_A'] + df['n_attn_B']
df['n_params_total'] = df['n_mlp_total'] + df['n_attn_total']

print("Spearman correlations: retention_A_abs vs circuit-size variables")
for col in ['n_mlp_A', 'n_mlp_B', 'n_mlp_total', 'n_attn_total', 'n_params_total']:
    rho, p = stats.spearmanr(df[col], df['retention_A_abs'])
    print(f"  vs {col:>14}: rho = {rho:+.3f}  p = {p:.4f}  {'*' if p < 0.05 else ''}")

# Per-method (since circuits per method differ wildly)
print("\nPer-method Spearman: retention_A_abs vs n_mlp_total")
for m in df['method'].unique():
    sub = df[df['method']==m]
    rho, p = stats.spearmanr(sub['n_mlp_total'], sub['retention_A_abs'])
    print(f"  {m:>11}: rho = {rho:+.3f}  p = {p:.4f}  n={len(sub)}")

# Scatter
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=True)
for ax, m in zip(axes, ['ACDC', 'ROMEtrace', 'Random']):
    sub = df[df['method']==m]
    if len(sub) == 0: continue
    for cond, color in [('shared', '#cc3333'), ('disjoint', '#2266aa')]:
        ss = sub[sub['condition']==cond]
        ax.scatter(ss['n_mlp_total'], ss['retention_A_abs'], color=color, alpha=0.7, label=cond, s=60, edgecolors='black', lw=0.4)
    ax.set_xlabel('n_mlp_A + n_mlp_B (circuit size sum)')
    ax.set_title(f'{m}'); ax.grid(alpha=0.3)
axes[0].set_ylabel('retention_A_abs')
axes[0].legend()
fig.suptitle('Does circuit size predict retention?', y=1.02)
fig.tight_layout(); fig.savefig(FIG_DIR / 'd2_circuit_size_vs_retention.png', dpi=140); plt.show()

### 6.4 Diagnostic 3 — Cross-method consistency

**Hypothesis:** Some pairs are simply harder than others, regardless of method. If true, retention should correlate ACROSS methods within a pair (i.e., when ACDC fails on pair X, ROMEtrace and Random also fail on pair X).

If this holds, the right framing isn't "method matters" — it's "pair difficulty matters", and our pair selection introduced an unmeasured confound.

In [ ]:
# Pivot: rows = pairs, cols = methods, values = retention
piv = df.pivot_table(index=['pair_idx','condition'], columns='method', values='retention_A_abs').reset_index()
print("Per-pair retention by method:")
print(piv.head(10))

# Pairwise correlation between methods
print("\nCross-method Spearman correlation of per-pair retention:")
method_cols = [c for c in ['ACDC', 'ROMEtrace', 'Random'] if c in piv.columns]
for i, m1 in enumerate(method_cols):
    for m2 in method_cols[i+1:]:
        valid = piv[[m1, m2]].dropna()
        if len(valid) > 5:
            rho, p = stats.spearmanr(valid[m1], valid[m2])
            print(f"  {m1:>11} vs {m2:>11}: rho = {rho:+.3f}  p = {p:.4f}  n={len(valid)}")

# Visualize: heatmap of per-pair retention
fig, ax = plt.subplots(figsize=(7, max(6, len(piv)*0.18)))
data = piv[method_cols].values
im = ax.imshow(data, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax.set_xticks(range(len(method_cols))); ax.set_xticklabels(method_cols)
ax.set_yticks(range(len(piv))); ax.set_yticklabels([f"p{r['pair_idx']:02d} ({r['condition'][:4]})" for _, r in piv.iterrows()], fontsize=8)
ax.set_title('Per-pair retention_A_abs by method  (green=retained, red=destroyed)')
plt.colorbar(im, ax=ax, label='retention_A_abs')
fig.tight_layout(); fig.savefig(FIG_DIR / 'd3_per_pair_heatmap.png', dpi=140); plt.show()

### 6.5 Diagnostic 4 — Why is KL super-additivity NEGATIVE?

**Observation:** super-additivity = `kl_postAB - (kl_A_solo + kl_B_solo)` is consistently negative (-13 to -32) across all conditions. Edits *cancel* drift rather than compound it.

**Possible mechanisms:**
- M1: Edit B's gradient partially undoes A's drift on neutral set (each edit pulls in opposite directions on average)
- M2: KL constraint dominates — by the end of edit B, the model is anchored close to original regardless of edits
- M3: Floor effect — kl_A_solo and kl_B_solo are individually inflated for some reason

**Test:** decompose. If kl_postAB ≈ max(kl_A_solo, kl_B_solo), it's M2 (anchoring). If kl_postAB << min, it's M1 (cancellation). If individual KL values are weirdly high, it's M3.

In [ ]:
df['kl_max_solo'] = df[['kl_A_solo','kl_B_solo']].max(axis=1)
df['kl_min_solo'] = df[['kl_A_solo','kl_B_solo']].min(axis=1)
df['kl_sum_solo'] = df['kl_A_solo'] + df['kl_B_solo']

print("KL decomposition (averages):")
print(f"  kl_A_solo:  {df['kl_A_solo'].mean():.2f} ± {df['kl_A_solo'].std():.2f}")
print(f"  kl_B_solo:  {df['kl_B_solo'].mean():.2f} ± {df['kl_B_solo'].std():.2f}")
print(f"  kl_postAB:  {df['kl_postAB'].mean():.2f} ± {df['kl_postAB'].std():.2f}")
print(f"  sum solo:   {df['kl_sum_solo'].mean():.2f}")
print(f"  max solo:   {df['kl_max_solo'].mean():.2f}")
print(f"  super_add:  {df['kl_super_additivity'].mean():.2f}  (= kl_postAB - sum_solo)")

# Diagnostic ratio: kl_postAB / max(kl_A_solo, kl_B_solo)
df['kl_postAB_vs_max_solo'] = df['kl_postAB'] / df['kl_max_solo'].clip(lower=0.1)
print(f"\nkl_postAB / max(kl_A_solo, kl_B_solo) average: {df['kl_postAB_vs_max_solo'].mean():.2f}")
print("  ~1.0 = M2 (anchoring); <1.0 = M1 (cancellation); >1.0 = compounding")

# Visualize
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(df['kl_sum_solo'], df['kl_postAB'], c=df['condition'].map({'shared':'#cc3333','disjoint':'#2266aa'}), alpha=0.7, s=55, edgecolors='black', lw=0.4)
lim = max(df['kl_sum_solo'].max(), df['kl_postAB'].max(), df['kl_max_solo'].max())
ax.plot([0, lim], [0, lim], 'k--', lw=0.8, label='additive (kl_postAB = sum)')
ax.plot([0, lim], [0, lim/2], 'g--', lw=0.8, alpha=0.5, label='half-sum (50% cancellation)')
ax.set_xlabel('kl_A_solo + kl_B_solo  (predicted if edits compounded)')
ax.set_ylabel('kl_postAB  (actual sequential drift)')
ax.set_title('Sequential KL drift vs solo sum — points BELOW diagonal = edits cancel')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / 'd4_kl_decomposition.png', dpi=140); plt.show()

### 6.6 Diagnostic 5 — Bimodal retention pattern

**Observation:** retention_A_abs distribution is heavily bimodal. Most rows are at 0.0 or 1.0, very few in between. This means "average retention" hides the real story.

**Test:** binarize retention into {kept (>0.5), destroyed (<0.5)} and ask: what predicts which mode a pair lands in?

In [ ]:
df['kept'] = (df['retention_A_abs'] > 0.5).astype(int)
print("Fraction of edits that survived sequential editing:")
print(df.groupby(['condition','method'])['kept'].agg(['sum','count','mean']).round(3).rename(columns={'sum':'n_kept','count':'n_total','mean':'kept_rate'}))

# Logistic-like: which features predict 'kept'?
from scipy.stats import pointbiserialr
print("\nFeature predictiveness for binary 'kept' outcome (point-biserial rho):")
for col in ['n_mlp_A', 'n_mlp_B', 'n_mlp_total', 'kl_A_solo', 'p_new_A_postA', 'p_new_A_solo']:
    if col not in df.columns: continue
    rho, p = pointbiserialr(df[col], df['kept'])
    print(f"  {col:>14}: rho = {rho:+.3f}  p = {p:.4f}  {'*' if p < 0.05 else ''}")

# Visualize: kept vs destroyed, what differs?
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col, label in zip(axes,
                          ['n_mlp_total', 'kl_A_solo', 'p_new_A_postA'],
                          ['n_mlp(A) + n_mlp(B)', 'kl_A_solo', 'p_new_A_postA (edit A success)']):
    kept = df[df['kept']==1][col]
    destroyed = df[df['kept']==0][col]
    ax.boxplot([destroyed.values, kept.values], labels=['destroyed', 'kept'], widths=0.6)
    ax.set_title(label); ax.grid(alpha=0.3, axis='y')
fig.suptitle('What predicts whether edit A survives edit B?', y=1.02)
fig.tight_layout(); fig.savefig(FIG_DIR / 'd5_kept_vs_destroyed.png', dpi=140); plt.show()

### 6.7 Final Summary — what these diagnostics tell us

In [ ]:
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
summary = {
    "notebook": "Notebook 6 — Diagnostic Analysis",
    "timestamp": ts,
    "input_rows": len(df),
    "d1_edit_A_success": {
        "frac_with_p_new_A_postA_above_05": float((df['p_new_A_postA'] > 0.5).mean()),
        "interpretation": "If <50%, the original null was a measurement-floor artifact: most edit-A's didn't even succeed."
    },
    "d2_circuit_size_predicts_retention": {
        "spearman_n_mlp_total_vs_retention": float(stats.spearmanr(df['n_mlp_total'], df['retention_A_abs'])[0]),
        "interpretation": "Negative correlation = larger circuits destroy retention. Positive = smaller circuits destroy it. Near-zero = circuit size isn't the explanatory variable."
    },
    "d4_kl_super_additivity_mechanism": {
        "mean_kl_postAB": float(df['kl_postAB'].mean()),
        "mean_sum_solo": float(df['kl_sum_solo'].mean()),
        "mean_max_solo": float(df['kl_max_solo'].mean()),
        "interpretation": "If kl_postAB ≈ max_solo, mechanism is M2 (KL anchoring dominates). If kl_postAB << min_solo, mechanism is M1 (gradient cancellation)."
    },
    "d5_binary_outcome": {
        "overall_kept_rate": float(df['kept'].mean()),
        "by_condition": df.groupby('condition')['kept'].mean().to_dict(),
        "by_method": df.groupby('method')['kept'].mean().to_dict(),
    }
}
with open(RESULTS_DIR / f"diagnostic_summary_{ts}.json", "w") as f:
    json.dump(summary, f, indent=2)
df.to_csv(RESULTS_DIR / f"diagnostic_dataframe_{ts}.csv", index=False)
print(json.dumps(summary, indent=2))
print(f"\nSaved -> {RESULTS_DIR}/diagnostic_summary_{ts}.json")
print(f"Saved -> {RESULTS_DIR}/diagnostic_dataframe_{ts}.csv")

# Bundle and download
import shutil
bundle = f"nb6_diagnostic_{ts}"
shutil.make_archive(bundle, 'zip', RESULTS_DIR)
print(f"\nBundle: {bundle}.zip")
from google.colab import files as colab_files
colab_files.download(f"{bundle}.zip")

### What to do with these results

Look at each diagnostic's output and decide which finding is strongest:

**If D1 shows <50% edit-A success rate:** 
Your null was a measurement artifact. Re-run the original H1 test on the clean subset; if H1 fires, that's your novel claim — *"Sequential interference is condition-dependent, but only for pairs where the first edit successfully landed. Prior work failing to filter for edit-A success would have missed this."*

**If D2 shows strong negative correlation:** 
Circuit size is the real predictor of sequential retention, not subject overlap. Novel claim: *"Sequential edit interference is governed by total circuit size at edits A and B, not by subject overlap. Larger circuits destroy retention regardless of localization signal."*

**If D3 shows high cross-method correlation:** 
Pair difficulty dominates method choice — same pairs fail across methods. Novel claim: *"Sequential edit difficulty is a pair-level property invariant to localization signal. This is consistent with editing relocating circuits in ways that no protection mechanism can anticipate."* (This is exactly what Notebook 7 will test directly.)

**If D4 shows kl_postAB ≈ max_solo:** 
The KL constraint is the dominant force, not the contrastive loss. Novel claim: *"Sequential KL drift is anchor-bounded rather than additive — the constraint dominates the gradient signal. This explains why localization choice doesn't separate methods: the KL term is doing all the work."*

**If D5 shows clean separators between kept and destroyed:** 
You have a predictive model for which sequential edits will succeed — useful as deployment guidance. Novel claim: *"Edit-A success probability and circuit size jointly predict whether a sequential edit will be retained, enabling a-priori filtering."*

Each of these is a defensible novel finding extractable from the existing v3 data. Pick the strongest after running and write the report around it.